In [ ]:
%pip install pandas
%pip install requests

In [ ]:
import pandas as pd

# read/load the .csv file
imdb_raw = pd.read_csv("imdb_data.csv")

# Extract and split the data
imdb_raw["title"] = imdb_raw["Name"].str.split("\n").str[0].str.strip()
imdb_raw["release_year"] = imdb_raw["Name"].str.extract(r"\((\d{4})\)")

# Select your required data and drop remaining data
imdb_clean = imdb_raw[["Rating", "title", "release_year"]].dropna()

# Convert year to integer for standard numeric matching
imdb_clean["release_year"] = imdb_clean["release_year"].astype(int)

# Save data to new .csv file
imdb_clean.to_csv("imdb_ratings.csv", index=False)
print("Saved! Columns are now: Rating, title, release_year")

In [ ]:
import pandas as pd

# read the csv file
data = pd.read_csv("netflix_titles.csv")

# print 5 data rows from top
print(data.head(), "\n\n")

# print 5 data rows from bottom
print(data.tail())

In [ ]:
# it works coz PANDAS can fetch the URL and parse JSON itself...
df = pd.read_json(
    "https://jsonplaceholder.typicode.com/posts"
)

# or if you have any "abc.json" file then you can directly call them inside read_json() method...
# data = pd.read_json(abc.json)
# print(data)

print(df.head())
print("\n\n", df.tail())

In [ ]:
# Better real world approach to call any API and transform data...
import requests
import json

# call an API to get json data
try:
    response = requests.get("https://jsonplaceholder.typicode.com/posts")
    response.raise_for_status()             # if response code is not 200 it throws an error.

    raw_data = response.json()
    # json.dumps() is used to convert data into JSON string.. ||| also json.dump() is there this are use to change python object into json format..
    # print(json.dumps(raw_data, indent=2)) # first check the api response how it looks like...

    df = pd.DataFrame(raw_data)             # __when api response is flat JSON then use DataFrames__

    # __when api response is nested/deep nested JSON then find the main key where actual data are coming, mostly the key which contain actual data name as "data"__
    # df = pd.json_normalize(raw_data["data"]["city"]) # __if you know more key then add after data__

    print(df.head())
    print("\n\n", df.tail())

except requests.exceptions.RequestException as e:
    print(f"Requests Error: {e}")

except ValueError as e:
    print(f"Value Error: {e}")

In [ ]:
# loc[] ==> Select rows and columns using labels (names).
df = pd.read_csv("netflix_titles.csv")
print(df.head(), "\n\n")
print(df.loc[0], "\n\n")                                        # Get the rows which index is 0
print(df.loc[[0,1,2]], "\n\n")                                  # Get multiple rows with the given index
print(df.loc[1, "title"], "\n\n")                               # Select specific rows and columns
print(df.loc[[1, 2], ["title", "type"]])                        # Select multiple columns

movie = df.loc[df["type"] == "Movie"].head()                    # filtering most imp. ( it's similar like WHERE type = 'Movie' )
print(movie)
movie2 = df.loc[df["type"] == "Movie", ["title", "country"]]    # if condition matches print mentioned columns...
print(movie2)

# multiple condition check:
filterDataOverReleaseYear = df.loc[
    (df["type"] == "Movie") & 
    (df["release_year"] > 2020)
]
print(filterDataOverReleaseYear)

filterDataOverCountry = df.loc[
    (df["type"] == "TV Show") &
    (df["country"] == "India")
]
print(filterDataOverCountry)

# Update data using loc[]:
df.loc[df["type"] == "Movie", "type"] = "Film"
print("Updated Data: ", df["type"])

# Get rows by range:
rangeData = df.loc[0:5] # it includes last data as well (Returns 6 rows)
print(rangeData)

# using "iloc[]"
data = df.iloc[1,2] # iloc[] uses position (.iloc[ row_position, column_position ]) WHERE ( .loc[] uses table )
print(data)


In [ ]:
# Practice Over Netflix data:
# Get all Netflix Movie Name:
movies = df.loc[df["type"] == "Movie", ["title", "director", "date_added"]]
print(movies)
movies2 = df.loc[df["type"] == "Movie", ["title", "director", "date_added"]].iloc[10:15]        # with iloc[] specific data
print(movies2)

# Get all TV shows:
tvShows = df.loc[df["type"] == "TV Show", ["title", "director", "date_added"]].iloc[0:5]
print(tvShows)

# get titles of indian content
indianContent = df.loc[df["country"] == "India", ["title", "director", "date_added"]].iloc[6:12]
print(indianContent)

# Get movies release after 2018
moviesAfter2018 = df.loc[
    (df["type"] == "Movie") &
    (df["release_year"] > 2018),
    ["title", "director", "release_year"]
].iloc[6:15]
print(moviesAfter2018)

# Show only title & release year
trYr = df.loc[:, ["title", "release_year"]]
print(trYr)

In [ ]:
# groupby()
import numpy as np

# Find number of movies & tv shows
numMoviesTv = df.groupby("type").size()
numMoviesTv = df.groupby(df["type"] == "Movie").size()        ## counts all like it includes NULL value as well
numMoviesTv = df["director"].value_counts(dropna=False)       ## counts excluding NULL or missing values (It completely excludes NULL or NaN)
print(numMoviesTv)
missing_directors = numMoviesTv[np.nan]                       ## with NumPy
# missing_directors = df["director"].isna().sum()             ## without NumPy
print("\nmissing_directors:", missing_directors)

print("---------------------------------")
# how many titles were released each year
relEacYr = df.groupby("release_year").size()       # for all (without any condition)
relEacYr = df[
    (df["type"] == "TV Show") &
    (df["director"].notnull())                      # notna() OR notnull()
].groupby("release_year").size()                    # With condition(s)
print(relEacYr)

print("__Production Style Clean__")
relEacYr = (
    df.loc[
        (df["type"] == "TV Show") &
        (df["director"].notnull())
    ]
    .groupby("release_year")
    .size()
)
print("\n", relEacYr)

In [ ]:
## Only groupby()

# --> Find the number of Movies and TV Shows.
num = df.groupby("type").size()
print(num)

print("------------------------")

# --> Find how many titles were released each year.
titles = df.groupby("release_year").size()
print(titles)

print("------------------------")

# --> Find the average release year for each type.
avg = df["release_year"].astype(int).groupby(df["type"]).mean().round(2)     # 1. Convert to int first, 2. Group by type, 3. Take the mean, 4. Then Round it
print(avg)

print("------------------------")

# --> Find top 10 countries with most Netflix content.

# explode() => take a list of data and split them on condition to a separate row(s)
# value_counts() => it counts number of times each unique value appears in a column and return in descending order...
top10 = df["country"].str.split(", ").explode().value_counts().head(10)
print(top10)

print("------------------------")

# --> Find how many Movies and TV Shows India has.
indieShow = df[df["country"].str.contains("India", na=False)].groupby("type").size()
print(indieShow)

In [ ]:
## Only Merge
df_rating = pd.read_csv("imdb_ratings.csv")
# remove duplicates
df_rating_clean = df_rating.drop_duplicates(subset=["title", "release_year"])

# --> Merge Netflix data with ratings data. ( show_id, title, type, Rating )
merge_df = df.merge(df_rating_clean, how="left", on=["title", "release_year"])
new_df = merge_df[["show_id", "title", "type", "Rating"]]

print(new_df)

print("-------------------------")

# --> Show only titles with ratings > 8.
data = new_df[new_df["Rating"] > 8]["title"]
print(data)

print("-------------------------")

# --> Find average rating by type.
avg_rating = merge_df.groupby("type")["Rating"].mean().round(4)
print(avg_rating)


In [ ]:
## Only sorting ( sort_values() )

# --> Sort by release year ascending.
assSortedData = df["release_year"].sort_values().head()
print(assSortedData)

print("-------------------------")

# --> Sort by release year descending.
descSortedData = df["release_year"].sort_values(ascending=False).head()
print(descSortedData)

print("-------------------------")

# --> Find the 20 newest Movies.
newestMovies = df[df["type"] == "Film"].sort_values(by="release_year", ascending=False).head(20)
print(newestMovies)

print("-------------------------")

# --> Find oldest TV Shows.
oldestTv = df[df["type"] == "TV Show"].sort_values(by="release_year").head(10)
print(oldestTv)

print("-------------------------")

# --> Sort by country and release year.
sortedData = df.sort_values(by=["country", "release_year"], ascending=[True, False])
print(sortedData)



In [ ]:
## Only remove duplicate ( drop_duplicates() )

# --> Check duplicate rows.
duplicates = df.drop_duplicates().sum()
print(duplicates)

print("-------------------------")

# --> Remove duplicate rows.
removeDuplicates = df.drop_duplicates()
print(removeDuplicates)

print("-------------------------")

# --> Check duplicate titles.
duplicateTitles = df["title"].duplicated().sum()
print(duplicateTitles)

print("-------------------------")

# --> Keep only unique titles.
uniqueTitles = df.drop_duplicates(subset=["title"])
print(uniqueTitles)

print("-------------------------")

# --> Find titles appearing more than once.
counts = df["title"].value_counts()
repeatedTitles = counts[counts > 1]
print(repeatedTitles)



In [ ]:
## Only isnull()

# --> Find null count in every column.
nullCount = df.isnull().sum()
print(nullCount)

print("-------------------------")

# --> Which column has most missing values?
# colMostMissingVal = df.isnull().sum().sort_values(ascending=False).head(1)      # to get column name and value
colMostMissingVal = df.isnull().sum().idxmax()                                    # to get only column name
print(colMostMissingVal)


print("-------------------------")

# --> Show rows where director is missing.
missingDirectors = df[df["director"].isnull()][["title", "director", "type", "release_year"]]
print(missingDirectors)

print("-------------------------")

# --> Show rows where country is missing.
missingCountries = df[df["country"].isnull()][["title", "country", "type", "release_year"]]
print(missingCountries)

print("-------------------------")

# --> Show rows where either country OR director is missing.
missingDataRows = df[df["country"].isnull() | df["director"].isnull()][["title", "country", "type", "director", "release_year"]]
print(missingDataRows)


In [ ]:
## Only isnull()

# --> Find null count in every column.
nullCount = df.isnull().sum()
print(nullCount)

print("-------------------------")

# --> Which column has most missing values?
# colMostMissingVal = df.isnull().sum().sort_values(ascending=False).head(1)      # to get column name and value
colMostMissingVal = df.isnull().sum().idxmax()                                    # to get only column name
print(colMostMissingVal)


print("-------------------------")

# --> Show rows where director is missing.
missingDirectors = df[df["director"].isnull()]
print(missingDirectors)

print("-------------------------")

# --> Show rows where country is missing.
missingCountries = df[df["country"].isnull()]
print(missingCountries)

print("-------------------------")

# --> Show rows where either country OR director is missing.
missingDataRows = df[df["country"].isnull() | df["director"].isnull()]
print(missingDataRows)


In [ ]:
## only fillna()

# --> Replace missing director names. ( use "unknown" )     # To replace permanently need to use that particular row eg: {{ df["director"] = df[.....]..... }}
unknown = df["director"].fillna("Unknown")
print(unknown)

print("-------------------------")

# --> Replace missing country with "Not Available".         # To replace permanently need to use that particular row eg: {{ df["country"] = df[.....]..... }}
missingCountry = df["country"].fillna("Not Available")
print(missingCountry)

print("-------------------------")

# --> Replace missing cast with "No Cast Data".             # To replace permanently need to use that particular row eg: {{ df["cast"] = df[.....]..... }}
missingCast = df["cast"].fillna("No Cast Data")
print(missingCast)

print("-------------------------")

# --> Fill missing rating values.
missingRating = df["rating"].fillna("No Rating").iloc[31:103]
print(missingRating)

print("-------------------------")

# --> Fill every null in dataset with "Missing".
allMissing = df.fillna("Missing")
print(allMissing)
